Importing Dependencies

In [14]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, ServiceContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from transformers import AutoTokenizer, AutoModelForCausalLM
import wikipediaapi
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.ollama import Ollama
from llama_index.core import Settings

Set up Wikipedia API and fetch articles

In [ ]:
wiki = wikipediaapi.Wikipedia("en")

animals = ["Elephant", "Cheetah", "Giraffe", "Dolphin", "Penguin"]

data = []
for animal in animals:
    page = wiki.page(animal)
    if page.exists():
        text = page.text
        data.append({"animal": animal, "text": text})
        with open(f"docs/{animal}.txt", "w", encoding="utf-8") as f:
            f.write(text)

Simple test with model

In [ ]:
import ollama

response = ollama.chat(model='llama2', messages=[
  {'role': 'user', 'content': 'Explain retrieval augmented generation in one sentence.'}
])

print(response['message']['content'])

Load the model in with its tokenizer

In [ ]:
# run 'ollama serve' and 'ollama llama2' in terminal before running this code
llm = Ollama(model="llama2")

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

Settings.num_output = 256
Settings.context_window = 3000

2025-11-01 22:11:19,582 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


/home/dsu/.local/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Helper function to generate responses for different chunk sizes

In [ ]:
def generate_responses(query):
    documents = SimpleDirectoryReader("docs").load_data()

    chunk_sizes = [128, 256, 512, 1024]

    for chunk_size in chunk_sizes:
        print(f"\n=== Chunk size: {chunk_size} ===")

        # Create a sentence-based text splitter
        splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=20) 
        nodes = splitter.get_nodes_from_documents(documents)

        Settings.llm = llm
        Settings.embed_model = embed_model

        index = VectorStoreIndex(nodes)
        query_engine = index.as_query_engine()

        response = query_engine.query(query)
        print("Response:", response.response)


Testing RAG retrieval for different queries

In [17]:
generate_responses("Explain how lions hunt together.")
# generate_responses("What do elephants eat?")
# generate_responses("Describe the habitat of penguins.")
# generate_responses("How fast can a cheetah run?")


=== Chunk size: 128 ===


2025-11-01 22:13:29,479 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"
2025-11-01 22:13:31,141 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Response: Lions are known to hunt together in coordinated attacks, using their unique vocalizations and body language to communicate and coordinate their efforts. During these hunts, lions work together to surround and isolate their prey, using their combined strength and agility to bring it down. The lions may also use their manes to signal their status and authority within the hunt, with dominant males taking on a more prominent role in leading the pack. By working together in this way, lions are able to take down larger and more formidable prey than they would be able to tackle alone, making hunting together an effective and efficient strategy for them.

=== Chunk size: 256 ===


2025-11-01 22:14:35,265 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Response: Cheetahs are known to be solitary hunters, but there have been observations of them cooperating with other predators in bringing down prey. One recorded instance involves cheetahs working with black-backed jackals to bring down a larger antelope. This suggests that while lions are primarily nocturnal and tend to hunt alone, they may also engage in coordinated hunting efforts, similar to cheetahs. However, without further information on the context of the query, it is difficult to provide a definitive answer. Can you provide more context or clarify the question?

=== Chunk size: 512 ===


2025-11-01 22:15:09,035 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Response: Lions are known to be social hunters, meaning they hunt in groups or prides. When hunting together, lions use a coordinated approach to catch their prey. Here's how they do it:

1. Scouting: Lions will often send out scouts to locate potential prey. These scouts will observe the prey from a distance and report back to the rest of the pride.
2. Stalking: Once the prey has been located, the lions will stalk it, moving quietly and stealthily to get close enough for an attack.
3. Ambush: The lions will then launch a coordinated ambush, with multiple lions attacking the prey from different angles. This allows them to catch their prey off guard and increase their chances of making a successful kill.
4. Communication: During the hunt, lions use vocalizations and body language to communicate with each other. They will often roar or growl to signal their location and intentions to the rest of the pride.
5. Teamwork: Lions are highly social animals, and they work together to bring down

2025-11-01 22:15:26,785 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Response: Lions are social animals that often hunt together in coordinated groups. During a hunt, multiple lions will work together to corner and bring down their prey. They use a variety of strategies to achieve this, including:

1. Stalking: Lions will stalk their prey, moving quietly and slowly until they are close enough to make a kill.
2. Communication: Lions use vocalizations and body language to communicate with each other during the hunt. They may let each other know when they have found prey, or when they need to be more careful.
3. Coordination: Lions will work together to corner their prey, using their size and strength to overpower it. They may also use coordinated attacks, with multiple lions attacking from different angles.
4. Support: Lions will often support each other during the hunt, providing a shoulder to lean on or a boost of confidence when needed.

Overall, hunting together is an important part of lion social dynamics and plays a key role in their survival and su